# Generating Supplemental Tables including engineering Motifs from INN Sequence Information



## Load DB Files ##

In [1]:
import sys
from pathlib import Path
from collections import Counter

import pandas as pd

In [2]:
repo_root = Path.cwd().resolve()
while not (repo_root / "src" / "inn_analysis_ops").is_dir():
    if repo_root.parent == repo_root:
        raise FileNotFoundError("Could not locate the repository root")
    repo_root = repo_root.parent

sys.path.insert(0, str(repo_root))
sys.path.insert(0, str(repo_root / "src"))
data_source_path_folder = repo_root / "data"
data_path = str(data_source_path_folder)
results_path = data_source_path_folder / "results"
results_path.mkdir(parents=False, exist_ok=True)

In [ ]:
from inn_analysis_ops.models.inn_structure import INNStructureCollection, INNStructureResult, INNChainDetail

Load database files for extracted INN data from disk 
For the correct expected format refer to 'docs/ExpectedDBFormatPydantic.md'


In [4]:

db_path = repo_root / "collection_dump.json"


In [ ]:
from inn_analysis_ops.models.inn_structure import INNStructureCollection
from inn_analysis_ops.data_io.load_db import load_from_json

collection: INNStructureCollection = load_from_json(db_path)


## Parse IMGT mutational information and realign if necessary

In [6]:
from inn_analysis_ops.models.numbering.conversion import create_numbering_object

NUMBERING_PATH = data_source_path_folder / "reference" / "IMGT_EU_NUMBERING.xlsx"
cn = create_numbering_object(NUMBERING_PATH)



In [7]:
for r in collection.results:
    for c in r.chains:
        if not hasattr(c, 'scraped_mutations'):
            if (not hasattr(c, "scraped_mutations") or c.scraped_mutations is None) and hasattr(c, "mutations"):
                c.scraped_mutations = c.mutations
        elif hasattr(c, 'mutations') and c.scraped_mutations is None:
            c.scraped_mutations = c.mutations


In [8]:
# load database of constant chain germline info

from inn_analysis_ops.models.inn_structure.constant_validator import validate_constant_domains

constant_db = data_source_path_folder / "reference" / "imgt_constant_regions.xlsx"
constant_db_seqs = pd.read_excel(constant_db, sheet_name="annotation_with_imgt_gaps")

res = validate_constant_domains(collection, constant_db_seqs)

In [9]:
# this step is run for realigning and tweaking hinge alignments. 
# that is find the best gene to a hinge wtih local alignment. 

from inn_analysis_ops.models.inn_structure.realigner import realign_constant_domains
from inn_analysis_ops.models.inn_structure.operations import update_with_realignment
from inn_analysis_ops.models.inn_structure.constant_validator import build_constant_db


cdb = build_constant_db(constant_db_seqs)
bundle = realign_constant_domains(res, collection=collection, cdb=cdb, expand_on_mismatch=True,allow_leading_insertion=False, allow_trailing_insertion=True)
# re update collection with the new alignments to hinge
collection = update_with_realignment(collection, bundle, min_match_alignment=0.2, cn=cn)

Counter(m.method for m in bundle.mutations), Counter(m.category for m in bundle.mutations)

(Counter({'positional': 3286, 'exact': 1078, 'smith-waterman': 752}),
 Counter({1: 4467, 2: 495, 3: 154}))

In [10]:
# this step will re align the CH1/CKAPPA/CLAMBDA regions for any substititons within a flank/nd domain

from inn_analysis_ops.models.inn_structure.realigner import realign_constant_domains_with_context

ch1_realignments = realign_constant_domains_with_context(
    collection=collection,
    bundle=bundle,
    target_regions={"CH1", "C-KAPPA", "C-LAMBDA"},
    min_match_alignment=0.2,
    cn=cn,
        cdb=cdb,  # required now

)
# re update collection with the new alignments to CH1
collection = update_with_realignment(
    collection,
    ch1_realignments.context_bundle_for_update,
    min_match_alignment=0.2,
    cn=cn,
)

(
    Counter(m.method for m in ch1_realignments.context_bundle_for_update.mutations),
    Counter(m.category for m in  ch1_realignments.context_bundle_for_update.mutations)
)

Context baseline mismatch inn=12168 chain=INN 12168_N detail=INN 12168_N.1 region=C-LAMBDA: stored_n_mut=2 recomputed_n_mut=3 (using recomputed for expansion)
Context baseline mismatch inn=12387 chain=INN 12387_N detail=INN 12387_N.1 region=C-LAMBDA: stored_n_mut=2 recomputed_n_mut=3 (using recomputed for expansion)
Context baseline mismatch inn=12457 chain=INN 12457_N detail=INN 12457_N.1 region=C-LAMBDA: stored_n_mut=2 recomputed_n_mut=3 (using recomputed for expansion)
Context baseline mismatch inn=13021 chain=INN 13021_N detail=INN 13021_N.1 region=C-LAMBDA: stored_n_mut=2 recomputed_n_mut=3 (using recomputed for expansion)
Context baseline mismatch inn=13020 chain=INN 13020_N detail=INN 13020_N.1 region=C-LAMBDA: stored_n_mut=2 recomputed_n_mut=3 (using recomputed for expansion)
Context baseline mismatch inn=13145 chain=INN 13145_M detail=INN 13145_M.1 region=CH1: stored_n_mut=3 recomputed_n_mut=4 (using recomputed for expansion)
Context baseline mismatch inn=13604 chain=INN 13604

(Counter({'smith-waterman': 109}), Counter({1: 109}))

In [11]:
# this step will re align the CH2regions for any substititons within a flank/nd domain
from inn_analysis_ops.models.inn_structure.realigner import realign_constant_domains_with_context

ch2_realignments = realign_constant_domains_with_context(
    collection=collection,
    bundle=bundle,
    target_regions={"CH2"},
    min_match_alignment=0.2,
    cn=cn,
        cdb=cdb,  # required now
)

# re update collection with the new alignments to CH2
collection = update_with_realignment(
    collection,
    ch2_realignments.context_bundle_for_update,
    min_match_alignment=0.2,
    cn=cn,
)

(
    Counter(m.method for m in ch2_realignments.context_bundle_for_update.mutations),
    Counter(m.category for m in  ch2_realignments.context_bundle_for_update.mutations)
)

Context baseline mismatch inn=8054 chain=INN 8054_H detail=INN 8054_H.3 region=CH2: stored_n_mut=3 recomputed_n_mut=2 (using recomputed for expansion)
Context baseline mismatch inn=11351 chain=INN 11351_H detail=INN 11351_H.3 region=CH2: stored_n_mut=7 recomputed_n_mut=6 (using recomputed for expansion)
Context baseline mismatch inn=11457 chain=INN 11457_H detail=INN 11457_H.7 region=CH2: stored_n_mut=3 recomputed_n_mut=8 (using recomputed for expansion)
Context baseline mismatch inn=13177 chain=INN 13177_H detail=INN 13177_H.3 region=CH2: stored_n_mut=7 recomputed_n_mut=6 (using recomputed for expansion)
Context baseline mismatch inn=11040 chain=INN 11040_H detail=INN 11040_H.3 region=CH2: stored_n_mut=3 recomputed_n_mut=2 (using recomputed for expansion)


(Counter({'smith-waterman': 550}), Counter({1: 550}))

In [12]:
# quick visual of realignment for specific INN chain/region annotation

for rm in ch2_realignments.context_bundle_for_update.mutations:
    if str(rm.inn_id) == "11038" and rm.chain_id == "INN 11038_H" and rm.region_annotation == "CH2":
        print(rm.visual_alignment(chunk=140, tick=10))
        break

INN 11038  CH2  (smith-waterman, ighg1*01)
IMGT :         10        20        30        40        50        60        70        80        90       100       110       120      
     :          |         |         |         |         |         |         |         |         |         |         |         |      
Ref  : ..PPEMLGGPSIFIFPPKPKDTLSI.SRTPEVTCLVVDLGPDDSDVQITWFVDNTQVY...TAKTSPREEQFN......STYRVVSVLPILHQDW..LKGKEFKCKVNSKS..LPSPIERTISKAK
Query: .....AA.A.....................................................................................................................
Mut  :      SS S                                                                                                                        S=sub D=del I=ins



In [13]:
result = next(r for r in collection.results if r.inn_id == "8495")
ch2 = next(c for c in result.chains if c.region_annotation == "CH2")

print("realigned_mutations:", ch2.realigned_mutations)
print("realigned_failed:", ch2.realigned_failed)
print("final_mutations:", ch2.final_mutations)
print("mutations_by_individual_numbering_systems:", ch2.mutations_by_individual_numbering_systems)

realigned_mutations: [MutationRecord(position=10, mutated_seq='S', germline_seq='P', mutation_type='substitution', insertion_str=None, mutation_region=None, repeat_region_count=None, numbering_system=None, original_position=None)]
realigned_failed: False
final_mutations: [MutationRecord(position=10, mutated_seq='S', germline_seq='P', mutation_type='substitution', insertion_str=None, mutation_region=None, repeat_region_count=None, numbering_system=None, original_position=None)]
mutations_by_individual_numbering_systems: [NumberedMutationRecord(position=MutationRecord(position=10, mutated_seq='S', germline_seq='P', mutation_type='substitution', insertion_str=None, mutation_region=None, repeat_region_count=None, numbering_system=None, original_position=None), imgt_position=MutationRecord(position=2, mutated_seq='S', germline_seq='P', mutation_type='substitution', insertion_str=None, mutation_region='CH2', repeat_region_count=0, numbering_system='IMGT', original_position=10), eu_position=M

In [14]:
result = next(r for r in collection.results if r.inn_id == "11385")
ch2 = next(c for c in result.chains if c.region_annotation == "HINGE-REGION")

print("realigned_mutations:", ch2.realigned_mutations)
print("realigned_failed:", ch2.realigned_failed)
print("final_mutations:", ch2.final_mutations)
print("mutations_by_individual_numbering_systems:", ch2.mutations_by_individual_numbering_systems)

realigned_mutations: [MutationRecord(position=5, mutated_seq='S', germline_seq='C', mutation_type='substitution', insertion_str=None, mutation_region=None, repeat_region_count=None, numbering_system=None, original_position=None)]
realigned_failed: False
final_mutations: [MutationRecord(position=5, mutated_seq='S', germline_seq='C', mutation_type='substitution', insertion_str=None, mutation_region=None, repeat_region_count=None, numbering_system=None, original_position=None)]
mutations_by_individual_numbering_systems: [NumberedMutationRecord(position=MutationRecord(position=5, mutated_seq='S', germline_seq='C', mutation_type='substitution', insertion_str=None, mutation_region=None, repeat_region_count=None, numbering_system=None, original_position=None), imgt_position=MutationRecord(position=5, mutated_seq='S', germline_seq='C', mutation_type='substitution', insertion_str=None, mutation_region='HINGE-IGHG1', repeat_region_count=0, numbering_system='IMGT', original_position=5), eu_positi

In [15]:
def compare_muts(mut_a, mut_b):
    if mut_a.position != mut_b.position:
        return False
    if mut_a.germline_seq != mut_b.germline_seq:
        return False
    if mut_a.mutated_seq != mut_b.mutated_seq:
        return False
    return True


evaluate_diffs = []
skipped_eval = []
for result in collection.results:
    for chain in result.chains:
        if chain.realigned_mutations is None:
            skipped_eval.append((result.inn_id, chain.chain_id, chain.region_annotation))
            continue

        imgt_mut_calls = sorted(chain.realigned_mutations or [], key=lambda m: (m.position, m.germline_seq, m.mutated_seq))
        realign_mut_calls = sorted(chain.realigned_mutations, key=lambda m: (m.position, m.germline_seq, m.mutated_seq))
        if imgt_mut_calls == realign_mut_calls:
            continue
        if len(imgt_mut_calls) == len(realign_mut_calls) and all(
            [compare_muts(m_a, m_b) for m_a, m_b in zip(imgt_mut_calls, realign_mut_calls)]
        ):
            continue

        if chain.region_annotation == "HINGE-REGION":
            continue
        # if chain.region_annotation == "CHS":
        # continue

        evaluate_diffs.append(
            {
                "inn_id": result.inn_id,
                "chain_id": chain.chain_id,
                "region": chain.region_annotation,
                "imgt_mutations": imgt_mut_calls,
                "realign_mutations": realign_mut_calls,
                "chain": chain,
            }
        )

In [16]:
bundle_mut = next(m for m in bundle.mutations if m.inn_id == "11678" and m.region_annotation == "HINGE-REGION")
print(bundle_mut.visual_alignment(chunk=100))

INN 11678  HINGE-REGION  (smith-waterman, ighg1*01)
IMGT :     5   10   15
     :     |    |    |
Ref  : EPKSCDKTHTCPPCP
Query: ....S.........^P
Mut  :     S         I   S=sub D=del I=ins



In [17]:
bundle_collection = collection

In [18]:
from inn_analysis_ops.models.inn_structure.operations import collapse_to_vh_vl_binder_info

arm_df = collapse_to_vh_vl_binder_info(bundle_collection, include_sequences_without_fv=True)
arm_df.to_csv(str(results_path / "imgt_2d_structure_arm_mut_info.csv"), index=False)
arm_df.head()

,inn_id,arm_slot,arm_binder_type,vh_domain_type,vl_domain_type,shared_vl,shared_heavy,duplicate_vh,vh,vh_v_gene,...,tcr_beta_gene,unique_tcr_beta_count,hinge_region,hinge_region_total_mutations,full_length_heavy,full_length_light,ch_chain_detail_order_idxs,cl_chain_detail_order_idxs,other_annotations_skipped,_no_fv
0,11211,a,canonical_heavy_kappa,ig_full_length,vkappa,False,False,False,QVQLQESGP.GLVKPSETLSLTCTVSGGSVS..SGDYYWTWIRQSP...,"IGHV4-61*01 (89.9%)(human), IGHV4-61*08 (89.9%...",...,NaN,NaN,EPKSCDKTHTCPPCP,NaN,QVQLQESGPGLVKPSETLSLTCTVSGGSVSSGDYYWTWIRQSPGKG...,DIQMTQSPSTLSASVGDRVTITCQASQDISNYLNWYQQKPGKAPKL...,INN 11211_H.1;INN 11211_H.2;INN 11211_H.3;INN ...,INN 11211_L.1,NaN,False
1,11218,a,canonical_heavy_kappa,ig_full_length,vkappa,False,False,False,QVQLVQSGA.EVKKPGASVKVSCKASGYTF....TTAGMQWVRQAP...,"IGHV1-3*01 (79.6%)(human), IGHV7-4-1*02 (79.6%...",...,NaN,NaN,EPKSCDKTHTCPPCP,NaN,QVQLVQSGAEVKKPGASVKVSCKASGYTFTTAGMQWVRQAPGQGLE...,DIQMTQSPSSLSASVGDRVTITCKASQDVSTAVAWYQQKPGKAPKL...,INN 11218_H.1;INN 11218_H.2;INN 11218_H.3;INN ...,INN 11218_L.1,NaN,False
2,11224,a,canonical_heavy_kappa,ig_full_length,vkappa,False,False,False,EVQLQESGP.GLVKPSETLSLTCTVSGFSL....TSNDIAWIRQPP...,IGHV4-4*08 (83.5%)(human)Alignment details,...,NaN,NaN,ESKYGPPCPPCP,1.0,EVQLQESGPGLVKPSETLSLTCTVSGFSLTSNDIAWIRQPPGKGLE...,DIVMTQTPLSLSVTPGQPASISCRSSQSLEYSDGYTYLEWYLQKPG...,INN 11224_H.1;INN 11224_H.2;INN 11224_H.3;INN ...,INN 11224_L.1,NaN,False
3,11328,a,canonical_heavy_kappa,ig_full_length,vkappa,False,False,False,QVTLKESGP.TLVKPTQTLTLTCSFSGFSLS..TSGMGIGWIRQPP...,IGHV2-5*09 (91.8%)(human)Alignment details,...,NaN,NaN,ESKYGPPCPPCP,1.0,QVTLKESGPTLVKPTQTLTLTCSFSGFSLSTSGMGIGWIRQPPGKA...,EIVLTQSPGTLSLSPGERATLSCTASSSVSSSYLHWYQQKPGKAPK...,INN 11328_H.1;INN 11328_H.2;INN 11328_H.3;INN ...,INN 11328_L.1,NaN,False
4,11198,a,canonical_heavy_kappa,ig_full_length,vkappa,False,False,False,QVQLKESGP.GLLAPSQSLSITCTVSGFSL....TGHGVNWVRQPP...,"IGHV2-6-7*01 (92.8%)(house mouse), IGHV2-6-7*0...",...,NaN,NaN,EPSGPISTINPCPPCKECHK,NaN,QVQLKESGPGLLAPSQSLSITCTVSGFSLTGHGVNWVRQPPGKGLE...,DVQMTQTTSSLSASLGDRVTITCRPSQDINNYLSWYQQKPDGTVKL...,INN 11198_H.1;INN 11198_H.2;INN 11198_H.4;INN ...,INN 11198_L.2,NaN,False


In [19]:
arm_df.query('inn_id == "11457"')

,inn_id,arm_slot,arm_binder_type,vh_domain_type,vl_domain_type,shared_vl,shared_heavy,duplicate_vh,vh,vh_v_gene,...,tcr_beta_gene,unique_tcr_beta_count,hinge_region,hinge_region_total_mutations,full_length_heavy,full_length_light,ch_chain_detail_order_idxs,cl_chain_detail_order_idxs,other_annotations_skipped,_no_fv
88,11457,a,swapped_chain_lambda,vh_ck,vl_ch1_fc,False,False,False,EVQLLESGG.GLVQPGGSLRLSCAASGFTF....STYAMNWVRQAP...,IGHV3-23*03 (87%)(human)Alignment details,...,NaN,NaN,NaN,NaN,EVQLLESGGGLVQPGGSLRLSCAASGFTFSTYAMNWVRQAPGKGLE...,EVQLLESGGGLVQPGGSLRLSCAASGFTFSDNAMGWVRQAPGKGLE...,INN 11457_H.10;INN 11457_H.5;INN 11457_H.7;INN...,INN 11457_L.2,A-DOMAIN,False
89,11457,b,canonical_heavy_kappa,ig_full_length,vkappa,False,False,True,EVQLLESGG.GLVQPGGSLRLSCAASGFTF....SDNAMGWVRQAP...,"IGHV3-23*01 (94.9%)(human), IGHV3-23D*01 (94.9...",...,NaN,NaN,EPKSCDKTHTCPPCP,NaN,EVQLLESGGGLVQPGGSLRLSCAASGFTFSDNAMGWVRQAPGKGLE...,EIVLTQSPGTLSLSPGERATLSCRASQSVSDEYLSWYQQKPGQAPR...,INN 11457_H.1;INN 11457_M.1;INN 11457_M.2;INN ...,INN 11457_N.1,NaN,False


## Merge scraping results across therasabdab, wilkinson hale, imgt


In [ ]:
from inn_analysis_ops.data_io.therasabdab import build_inn_table, build_sequence_table, TheraSAbDabCsvSchema
from inn_analysis_ops.data_io.imgt_mab_dab import IMGTMabDbCsvSchema
from inn_analysis_ops.data_io.wilk_hale import  WilkHaleCsvSchema


thera_db_path = repo_root / "therasabdab.csv"
imgt_mabd_dab_path = repo_root / "IMGT_mabdb.csv"
wilk_hale_path = repo_root / "wilk_hale_inn_analysis.csv"

In [21]:
import pandas as pd
from pandera.typing import DataFrame 


thera_df: DataFrame[TheraSAbDabCsvSchema] = TheraSAbDabCsvSchema.validate(pd.read_csv(thera_db_path, index_col=0))

inn_table_thera = build_inn_table(thera_df).to_dataframe()
seq_table_thera = build_sequence_table(thera_df).to_dataframe()

In [22]:

imgt_df: DataFrame[IMGTMabDbCsvSchema] = IMGTMabDbCsvSchema.validate(pd.read_csv(imgt_mabd_dab_path, index_col=0))
wilk_hale_df: DataFrame[WilkHaleCsvSchema] = WilkHaleCsvSchema.validate(pd.read_csv(wilk_hale_path))


In [23]:
from inn_analysis_ops.data_io.merge import DatabaseMerger
from inn_analysis_ops.data_io.merge import update_imgt_df_using_merged

merged = DatabaseMerger.from_collapsed(
    thera_inn_df=inn_table_thera,
    thera_seq_df=seq_table_thera,
    imgt_df=imgt_df,
    mabsolve_df=wilk_hale_df,
    imgt_2d_arm_df=arm_df,
)

imgt_df_updated = update_imgt_df_using_merged(imgt_df, merged)

In [24]:
from inn_analysis_ops.data_io.merge import filter_inn_df

all_df = merged.inn.to_dataframe()
kept, removed = filter_inn_df(all_df)

print(f"Kept: {len(kept)}, Removed: {len(removed)}")
removed["removal_reason"].value_counts()

Kept: 1748, Removed: 8


removal_reason
no_inn_number_not_in_therasabdab    5
no_inn_number_in_therasabdab        3
Name: count, dtype: int64

In [25]:
imgt_inn_number_to_inn_map = all_df[["inn_number", "inn_name"]].dropna().set_index("inn_number").to_dict()["inn_name"]
pd.Series(imgt_inn_number_to_inn_map).value_counts().max() == 1
imgt_inn_number_to_inn_map = {
    **imgt_inn_number_to_inn_map, 
    **{int(float(key)): value for key, value in imgt_inn_number_to_inn_map.items() if isinstance(key, str) and key.replace('.', '', 1).isdigit()},
    **{str(int(float(key))): value for key, value in imgt_inn_number_to_inn_map.items() if isinstance(key, str) and key.replace('.', '', 1).isdigit()}
}

In [26]:
inn_mol_info = merged.inn.to_dataframe()


no_vh_vl_constant_seqs_found_anywhere = inn_mol_info[inn_mol_info[['n_binders_therasabdab', 'n_binders_imgt']].isnull().all(axis=1)]['inn_name'].unique()#[['inn_name', 'n_binders_therasabdab', 'n_binders_imgt']].fillna(0).query('n_binders_imgt==0')['inn_name'].unique()
no_imgt_seq_info = inn_mol_info[inn_mol_info[['n_binders_imgt']].isnull().all(axis=1)]['inn_name'].unique()#[['inn_name', 'n_binders_therasabdab', 'n_binders_imgt']].fillna(0).query('n_binders_imgt==0')['inn_name'].unique()
no_imgt_seq_info_names = set(no_imgt_seq_info) - set(no_vh_vl_constant_seqs_found_anywhere)


In [27]:
len(no_imgt_seq_info_names)

10

In [28]:
all_sequence_info_found = merged.sequences.to_dataframe()


In [29]:
no_imgt_but_seq_elsewhere_data_subset = inn_mol_info.query("inn_name.isin(@no_imgt_seq_info_names)")
no_imgt_but_seq_elsewhere_data_subset['removal_reason'] = 'imgt sequence info not found'
no_imgt_but_seq_elsewhere_data_subset_new_seqs = set(no_imgt_but_seq_elsewhere_data_subset['inn_name']) - set(removed['inn_name'])
no_imgt_but_seq_elsewhere_data_subset = no_imgt_but_seq_elsewhere_data_subset.query("inn_name.isin(@no_imgt_but_seq_elsewhere_data_subset_new_seqs)")

no_vh_vl_data = inn_mol_info.query("inn_name.isin(@no_vh_vl_constant_seqs_found_anywhere)")
no_vh_vl_data['removal_reason'] = 'no vh/vl/constant chain sequences found'
no_vh_vl_data_new_seqs = set(no_vh_vl_data['inn_name']) - set(removed['inn_name'])
no_vh_vl_data = no_vh_vl_data.query("inn_name.isin(@no_vh_vl_data_new_seqs)")


assert len(set(no_vh_vl_data.reset_index()['inn_name']) & set(no_imgt_but_seq_elsewhere_data_subset.reset_index()['inn_name'])) == 0

removed_including_no_seq_info = pd.concat([
    removed, 
    no_imgt_but_seq_elsewhere_data_subset,
    no_vh_vl_data
])

assert removed_including_no_seq_info['inn_name'].value_counts().max() == 1

In [30]:
removed_including_no_seq_info.shape

(132, 59)

In [31]:
report_vh_vl_in_removed_seqs = (
    all_sequence_info_found.loc[
        all_sequence_info_found["inn_name"].isin(
            removed_including_no_seq_info["inn_name"]
        )
    ]
    .groupby(by="inn_name")
    .agg({"vh": lambda vh: ";".join(vh), "vl": lambda vl: ";".join(vl)})
)
removed_including_no_seq_info = removed_including_no_seq_info.drop(
    ["vh", "vl"], axis=1, errors="ignore"
).join(report_vh_vl_in_removed_seqs, on="inn_name", how="left")
all_sequence_info_found = all_sequence_info_found[
    ~all_sequence_info_found["inn_name"].isin(
        removed_including_no_seq_info["inn_name"]
    )
].copy()

inn_mol_info_without_exclusions = inn_mol_info[
    ~inn_mol_info["inn_name"].isin(
        removed_including_no_seq_info["inn_name"]
    )
]
assert set(inn_mol_info_without_exclusions["inn_name"]) == set(
    all_sequence_info_found["inn_name"]
)

len(inn_mol_info_without_exclusions["inn_name"].unique()), len(
    all_sequence_info_found["inn_name"].unique()
)

(1624, 1624)

In [32]:
inn_mol_info.to_csv(str(results_path /"all_source_inn_info.csv"), index=False)
all_sequence_info_found.to_csv(str(results_path / "all_source_sequences.csv"), index=False)

In [33]:
from inn_analysis_ops.models.inn_structure.analytics import parse_collection, build_constant_chain_pivot


chains, full_length, mutations_constant_chain = parse_collection(
    bundle_collection.results,
    inn_map=imgt_inn_number_to_inn_map
)

observed_chains = chains.to_dataframe()
observed_full_length_df = full_length.to_dataframe()
mutational_profile_constant_chain_df = mutations_constant_chain.to_dataframe()
merged_by_chain = build_constant_chain_pivot(observed_chains)
print(f"observed_chains:      {len(chains.records)} rows, {observed_chains.shape[1]} cols")
print(f"observed_full_length: {len(full_length.records)} rows")
print(f"mutational_profile:   {len(mutations_constant_chain.records)} rows")

observed_chains:      16963 rows, 26 cols
observed_full_length: 3154 rows
mutational_profile:   3326 rows


In [34]:
assert mutational_profile_constant_chain_df.index.duplicated().sum() == 0

In [35]:
assert merged_by_chain.index.duplicated().sum() == 0

In [ ]:
# check all sequences that have no mutations based on observed_full_length_df
full_len_no_mut = set(
    observed_full_length_df.groupby(by=["inn"], dropna=False)[
        "total_constant_mutations"
    ]
    .apply(lambda x: x.fillna(0).sum())
    .reset_index()
    .query("total_constant_mutations ==0")["inn"]
    .unique()
)  # ['total_constfant_mutations'].value_counts(dropna=False)
len(full_len_no_mut)

In [37]:
# check all sequences in arm df that have no mutations. or nothing logged for ch/cl mutations
arm_df_no_mut = set(
    all_sequence_info_found.groupby(by=["inn_name"])[
        ["imgt_ch_mutations", "imgt_cl_mutations"]
    ]
    .apply(lambda x: (~x.isnull()).sum())
    .sum(axis=1)
    .reset_index()
    .rename(columns={0: "count"})
    .query("count==0")["inn_name"]
)

In [ ]:
print(len(full_len_no_mut), len(arm_df_no_mut))
assert len(full_len_no_mut -arm_df_no_mut) == 0
assert len(arm_df_no_mut - full_len_no_mut) == 0


In [ ]:
# filter out any mutations taht we have now removed
sequences_to_skip_or_without_mutations = full_len_no_mut | set(removed_including_no_seq_info['inn_name'])
mutational_profile_constant_chain_df = mutational_profile_constant_chain_df.query("~inn.isin(@sequences_to_skip_or_without_mutations)").copy()
mutational_profile_constant_chain_df.shape, len(set(mutational_profile_constant_chain_df.index.get_level_values('inn')))

In [40]:
rm_inn = set(removed_including_no_seq_info['inn_name'])
in_ob_but_not_cc = set(observed_full_length_df['inn']) - set(merged_by_chain.index.get_level_values('inn'))
assert  len(set(merged_by_chain.index.get_level_values('inn')) - set(observed_full_length_df['inn'])) == 0

In [41]:
len(set(merged_by_chain.index.get_level_values('inn')))

1461

In [42]:
inns_no_mutations_but_have_constant = full_len_no_mut - in_ob_but_not_cc
inns_with_mutations = set(mutational_profile_constant_chain_df.index.get_level_values('inn'))
assert len(inns_no_mutations_but_have_constant & inns_with_mutations) == 0
assert (inns_no_mutations_but_have_constant | inns_with_mutations) == set(merged_by_chain.index.get_level_values('inn'))


In [43]:
len(inns_with_mutations)

750

In [44]:
from inn_analysis_ops.models.inn_structure.analytics import motif_profile_table

group_level = ["inn", "chain_id", "chain_type", "gene", "gene_species", "region_iteration", "region_classification"]
motifs_extracted_table = motif_profile_table(mutational_profile_constant_chain_df, group_level)
assert set(motifs_extracted_table.index.get_level_values('inn')) == inns_with_mutations
assert motifs_extracted_table.index.duplicated().sum() == 0

In [45]:
motif_patterns = (
    motifs_extracted_table.reset_index()
    .groupby(by=["gene", "region_classification", "constant_chain_mutational_motif"], dropna=False)
    .agg({"EU": "first", "mutation_type": "first", "germline_seq": "size", "num_mutations": "first", "inn": lambda x: sorted(set(x))})
    .sort_values(by="germline_seq", ascending=False)
    .rename(columns={"germline_seq": "num_total_mutation_observations"})  # counts all instances of a mutation even if its on a multi chain molecule
    .reset_index()
)
motif_patterns['num_unique_inn'] = motif_patterns['inn'].apply(lambda x: len(x))  # only considers unique INNs containing the mutation type
motif_patterns_export = motif_patterns.copy()
motif_patterns_export['inn'] = motif_patterns_export['inn'].apply(lambda x: ','.join(x))

In [46]:
unique_motif_patterns_per_inn = motifs_extracted_table.reset_index().groupby(by=["gene", "region_classification", "constant_chain_mutational_motif"], dropna=False)['inn'].value_counts().reset_index().set_index('inn').sort_index().rename(
    columns={'count': 'num_chains_with_mutation'}
)
unique_motif_patterns_per_inn_export =unique_motif_patterns_per_inn.copy()


In [47]:

with pd.ExcelWriter(str(results_path / "all_sources_updated.xlsx"), engine="openpyxl") as w:
    imgt_df_updated.to_excel(w, sheet_name="imgt_inn", index=False)
    arm_df.to_excel(w, sheet_name="imgt_sequences", index=False)
    inn_table_thera.to_excel(w, sheet_name="therasabdab_inn", index=False)
    seq_table_thera.to_excel(w, sheet_name="therasabdab_sequences", index=False)
    wilk_hale_df.to_excel(w, sheet_name="wilk_hale", index=False)

In [ ]:

with pd.ExcelWriter(str(results_path / "inn_analysis.xlsx"), engine="openpyxl") as w:
    # inns summary info => where we found at least a vh/vl/constant chain info
    inn_mol_info_without_exclusions.to_excel(w, sheet_name="kept_inn", index=False,merge_cells=False)
    # worksheet related to what was excluded form analysis
    removed_including_no_seq_info.to_excel(w, sheet_name="removed_inn", index=False,merge_cells=False)
    # inn with the vh/vl/ch/cl sequence info
    all_sequence_info_found.to_excel(w, sheet_name="inn_ab_sequence_info", index=False,merge_cells=False)
    # inn where we annotate each individual chain id
    observed_full_length_df.to_excel(w, sheet_name="IMGT Info Per Chain", index=False,merge_cells=False)
    # inn of only constant chain sequenc einformation
    merged_by_chain.to_excel(w, sheet_name="INNConstantChains", index=True, merge_cells=False)
    # CORE mutation table => individual mutations found in each inn with at least one mutation
    mutational_profile_constant_chain_df.to_excel(w, sheet_name="IndividualMutationProfile", index=True,merge_cells=False)
    # concenation of mutations across an inn chain
    motifs_extracted_table.reset_index().to_excel(w, sheet_name="ConstantChainMotifsByINNRegion", index=False,merge_cells=False)
    # consolidated view of motifs extracted table. Remove duplicated mutations across chains of an INN
    # NOTE: Excel worksheet names must be <= 31 chars.
    unique_motif_patterns_per_inn_export.to_excel(w, sheet_name="UniqueChainMotifsByINNRegion", index=False,merge_cells=False)
    # all observed set of mutations in constant chain
    motif_patterns_export.to_excel(w, sheet_name="Constant ChainMutationMotifs", index=False,merge_cells=False)
    merged.inn.to_schema_dataframe().to_excel(w, sheet_name="inn_schema", index=False,merge_cells=False)
    merged.sequences.to_schema_dataframe().to_excel(w, sheet_name="sequence_schema", index=False,merge_cells=False)

removed_including_no_seq_info["removal_reason"].value_counts()


removal_reason
no vh/vl/constant chain sequences found    117
imgt sequence info not found                 7
no_inn_number_not_in_therasabdab             5
no_inn_number_in_therasabdab                 3
Name: count, dtype: int64

# Generate Supplemental Table Files

In [ ]:
from inn_analysis_ops.models.inn_structure.wt_lookup import build_germline_numbering_table
import pandas as pd

germline_table = build_germline_numbering_table(_cdb, _cn)

out_file = results_path / "SuppTableNumbering_MapEuToIMGT.xlsx"
with pd.ExcelWriter(out_file, engine="openpyxl") as writer:
    _cn.filled_position_df.to_excel(writer, sheet_name="numbering_map")
    germline_table.to_excel(writer, sheet_name="germline_eu_assignments", index=False)

print(f"Wrote {out_file}")

In [52]:
from inn_analysis_ops.data_io.merge.loader import load_inn_analysis
from inn_analysis_ops.models.inn_structure.constant_validator import (
    build_constant_db,
)
from inn_analysis_ops.models.numbering.core import (
    ConstantChainNumbering,
    get_imgt_constant_numbering_map_table,
)
from inn_analysis_ops.data_io.merge.loader import  MutationPatternIssue




# Build the two dependencies once (same as before)
_cdb = build_constant_db(
    pd.read_excel(
        data_source_path_folder / "reference" / "imgt_constant_regions.xlsx",
        sheet_name="annotation_with_imgt_gaps",
    )
)
_cn = ConstantChainNumbering()
for _sheet in ["VH", "IGL", "IGK", "VH-HINGE"]:
    _cn.add_region_pos_map(
        get_imgt_constant_numbering_map_table(
            data_source_path_folder / "reference" / "IMGT_EU_NUMBERING.xlsx", _sheet
        )
    )

# Then load db data saved as excel from above with the fixer in one call
ds = load_inn_analysis(
    results_path / "inn_analysis.xlsx",
    cdb=_cdb,
    cn=_cn,
    mutation_fixer=[MutationPatternIssue.DUPLICATE_ROWS, MutationPatternIssue.MISSING_GERMLINE_SEQ]
)



In [53]:
from tqdm import tqdm
from inn_analysis_ops.analytics.motifs.pipelines.chain import chain_collection_to_enumeration_rows,ChainEnumerationRowAnnotated

db_sequences_mutation_info: list[ChainEnumerationRowAnnotated] = [
    seq_info
    for seq_info in tqdm(
        chain_collection_to_enumeration_rows(
            ds.full_length_chain_collection
        ),
        total=len(ds.full_length_chain_collection.records),
        desc="Enumerating chains",
    )
    if seq_info.total_constant_mutations > 0
]
mutations_to_ignore_for_motif_ops = {
    "insertion": {"HINGE": ["230", "230.i", "219", "220"], "FC": ["447"]},
    "deletion": {
        "FC": ["447"],
        "HINGE": [
            "230",
            "230.i",
            "217",
            "216",
            "218",
            "215",
            "219",
            "220",
        ],
    },
}


Enumerating chains:  29%|██▉       | 918/3154 [00:00<00:00, 880890.20it/s]


In [54]:
# add motif annotation info

from inn_analysis_ops.analytics.motifs.manual_curation import (
    build_effector_motif_pe_dataframe,
    build_effector_single_motifs_dataframe,
    update_effector_motif_pe_dataframe_with_single_calls,
    add_gene_allele_info,
    parse_motif_label_pe,
)

effector_motif_df_pre_single_merge = build_effector_motif_pe_dataframe(
    Path(data_path) / "motif_annotation" / "motif_classification_summary.csv"
)
effector_single_mut_df = build_effector_single_motifs_dataframe(
    Path(data_path) / "motif_annotation" / "single_mutation_classification_calls.xlsx",
    melt_data=False,
)


# re run again but make it less strict for the reamining
# rows that were not classified
effector_single_mut_df = add_gene_allele_info(
    effector_single_mut_df,
    inn_summary=ds.individual_mutation_profile,
    join_on_mutated_seq=False,
)

effector_motif_df = update_effector_motif_pe_dataframe_with_single_calls(
    effector_motif_df_pre_single_merge,
    effector_single_mut_df,
)
effector_motif_df.loc[
    effector_motif_df["mutation_string"].isnull(), "mutation_string"
] = effector_motif_df.loc[
    effector_motif_df["mutation_string"].isnull(),
    "mutation_string_pe",
].map(parse_motif_label_pe)


effector_motif_df = effector_motif_df[
    [
        "mutation_string",
        "mutation_string_pe",
        "size",
        "group",
        "sub_classification",
        "allele",
        "gene",
        "gene_species",
    ]
]
effector_motif_df["size"] = effector_motif_df["mutation_string"].apply(
    lambda x: len(x.split("/"))
)

effector_motif_df = add_gene_allele_info(
    effector_motif_df,
    inn_summary=ds.individual_mutation_profile,
    join_on_mutated_seq=False,
)
effector_motif_df[["gene", "gene_species", "allele"]] = (
    effector_motif_df[["gene", "gene_species", "allele"]]
    .fillna("")
    .map(lambda x: x.upper())
)
effector_motif_df.to_csv(str(results_path / "effector_motif_pe_updated_cc.csv"))

Deleting 2 rows with bad mutation_string: ['G137E/N203D/K/R214T/H268Q/K274Q/R355Q/Q419E', 'G107Adel/K207R/P211L/S215-']


In [55]:
from inn_analysis_ops.analytics.motifs.manual_curation import _load_imgt_motif_curation
imgt_biotechnology_manual_motif_curation =_load_imgt_motif_curation(
    Path(data_path) / "motif_annotation" / "IMGTBiotechnologyEffectorClassifications.csv"
)

ERROR CANNOT PARSE K


In [56]:
from inn_analysis_ops.analytics import (
    ConstantChainAnalysisConfig,
    build_constant_chain_analysis,
    export_constant_chain_analysis,
    CoMutationConfig,
    MotifIndexConfig,
    EffectorAnnotationConfig,
)

ab_seq_info_cols = [
    "inn",
    "inn_id",
    "chain_id",
    "chain_type",
    "sequence",
    "variable_chain_species",
    "constant_chain_species",
    "chain_description",
    "germline_sequence",
    "v_allele",
    "j_allele",
    "c_allele",
    "total_mutations",
    "total_constant_mutations",
    "total_mutations_unclassified_features",
    "variable_mutations",
    "constant_chain_mutations_IMGT",
    "constant_chain_mutations_EU",
    "constant_chain_mutations_KABAT",
]


inn_col_info = [
    "inn_name",
    "inn_number",
    "imgt_id",
    "format",
    "receptor_type",
    "isotype",
    "year_proposed",
    "year_recommended",
    "year_received",
    "specificity",
    "Species",
    "mechanism_of_action",
    "clinical_indication",
    "clinical_domain",
    "effector_function",
    "half_life",
    "heterodimerization",
    "site_specific_conj",
    "stabilized",
    "c_term_truncation",
    "other_fc_mod",
    "sources",
]
MUTATION_FREQ_RESOLUTION = ["gene_species", "gene"]
mutations_to_ignore_for_motif_ops = {
    "insertion": {"HINGE": ["230", "230.i", "219", "220"], "FC": ["447"]},
    "deletion": {
        "FC": ["447"],
        "HINGE": [
            "230",
            "230.i",
            "217",
            "216",
            "218",
            "215",
            "219",
            "220",
        ],
    },
}

cfg = ConstantChainAnalysisConfig(
    mutation_freq_resolution=MUTATION_FREQ_RESOLUTION,
    mol_info_cols=inn_col_info,
    sequence_cols=ab_seq_info_cols,
    inn_date_source="min_year",
    mutation_types=['substitution', 'deletion', 'insertion'],
    co_mutation=CoMutationConfig(
        min_count=2,
        position_col="EU",
        keep_classifications=[
            "intra",
            "inter_same_gene",
            "inter_cross_gene",
        ],
    ),
    motif_index=MotifIndexConfig(
        min_count=2,
        mutations_to_ignore_for_motif_ops=mutations_to_ignore_for_motif_ops,
        max_pvalue=0.05,
        
        min_lift=1.0,
        cohesion_frac_threshold=0.6,        
        max_cohesion_iterations=0,
        hard_stat_filters=["pvalue", "lift"], # "cohesion"],
        additional_user_set_filter_terms={},
        allowed_motifs_by_select=[
            'bispecific independent arm',
            # 'bispecfic shared anchor', 
            'manual_motif_curation'
        ],
        debug_step=None,
        manual_motif_definitions=imgt_biotechnology_manual_motif_curation.query('motif_size>1')['mutation_string_pe'].tolist()

    ),
    motif_classification=EffectorAnnotationConfig(
        effector_df=effector_motif_df,
        annotation_cols=["group", "sub_classification"],
    ),
)


In [ ]:
from inn_analysis_ops.analytics import (
    ConstantChainAnalysisConfig, build_constant_chain_analysis,
    export_constant_chain_analysis, CoMutationConfig, MotifIndexConfig,
    EffectorAnnotationConfig,
    build_chain_motif_index,   # ← new
    ChainMotifIndexResult,     # ← new
)
import logging
from notebooks._widget_run import run_with_split_output
import logging
logging.getLogger("analytics.co_mutation_analytics").setLevel(logging.DEBUG)
logging.basicConfig(level=logging.DEBUG, force=True)
result = run_with_split_output(build_constant_chain_analysis, ds, cfg) # cached_motif_result=result_old.motif_result)

In [ ]:
# Export all sheets (default)
import re
import re

export_constant_chain_analysis(
    result,
    str(results_path / "SupplementalData.xlsx"),
    column_serializers={
        # global — inn_set appears across multiple sheets
        "inn_set": lambda x: ", ".join(sorted(x)) if isinstance(x, (list, set)) else x,
        # per-sheet for CoMutationPairs
        ("CoMutationPairs", "inn_set_A"): lambda x: len(x) if isinstance(x, set) else x,
        # source label normalisation — global
        "sources": lambda x: (
            re.sub("mabsolve", "wilkinson/hale", x, flags=re.IGNORECASE)
            if isinstance(x, str)
            else x
        ),
        ("CoMutationPairs", "inn_count_by_gene_position_1"): lambda x: (
            ";".join(f"{gene}: {count}" for gene, count in x.items())
            if isinstance(x, dict)
            else eval(x)
            if isinstance(x, str)
            else x
        ),
        ("CoMutationPairs", "inn_count_by_gene_position_2"): lambda x: (
            ";".join(f"{gene}: {count}" for gene, count in x.items())
            if isinstance(x, dict)
            else eval(x)
            if isinstance(x, str)
            else x
        ),
    },
    column_renames={
        "species": "Species",
        (
            "ConstantMutSummary",
            "CH1/C-KAPPA/C-LAMBDA",
        ): "Total Mutations: CH1/C-KAPPA/C-LAMBDA",
        ("ConstantMutSummary", "HINGE-REGION"): "Total Mutations: HINGE-REGION",
        ("ConstantMutSummary", "CH2"): "Total Mutations: CH2",
        ("ConstantMutSummary", "CH3"): "Total Mutations: CH3",
        ("ConstantMutSummary", "CH4"): "Total Mutations: CH4",
        ("ConstantMutSummary", "CHS"): "Total Mutations: CHS",
        "gene_species": "Species",
        "mutation_string_pe": "Motif",
        "constant_chain_mutational_motif": "Motif",
        ("MotifIndex", "inn_count"): "Total Unique INNs",
        ("MotifIndex", "adjusted_count"): "Total Unique INNs (excl. supermotif)",
        ("MutRepetitionsPerINN", "mutation_string_pe"): "Mutation",
        ("UniqueSingleMutations", "mutation_string_pe"): "Mutation",
    },
    drop_columns={"all": ["mutation_string"]},
    include_columns={
        "MoleculeInfo": inn_col_info,
        "SkippedINNs": inn_col_info + ['removal_reason','vh','vl'],
        "MotifIndex": [
            "mutation_string_pe",
            "gene",
            "gene_species",
            "inn_count",
            "adjusted_count",
            "group",
            "sub_classification",
            "num_mutations",
            "sub_motifs",
            "novel_mutations",
            "freq_motif",
            "marginal_count",
            "expected_freq_wrt_indep_mutations",
            "expected_freq_wrt_sub_motifs",
            "log_lift_wrt_sub_motifs",
            "pvalue_wrt_sub_motifs",
            "pvalue_wrt_indep_mutations",
            "most_sig_sub_motif",
            "longest_motif_in_group",
            "larger_motif_with_same_inns",
            "inns",
            "adjusted_inns",
            "min_year",
            "max_year",
            "date_range",
            "cluster_seeds",
            "n_clusters",
            "primary_cluster_seeds",
            "primary_cluster_rep",
        ],
    },
    include_sheets=[
        "MoleculeInfo",  # inn info
        "SkippedINNs",  # skipped inns
        "FullSequenceList",  # all sequence found per inn
        "ConstantMutSummary",  # lists which inns have constant chain mutations
        "UniqueSingleMutations",  # aggregated list of most commonly occuring single residue mutations
        "MotifIndex",  # motif search algorithm result
        "PositionalMutFreq",  # aggregated list of all muations at a position
        "PairedChainMotifs", # paired motifs in the same inn
        "CoMutationPairs",  # statistics for co-mut frequency / pairwise muts
        "MutRepetitionsPerINN",  # unique  mutations/ concatenatned list of mutations oberved (not motifs yet)
        "InsertionMutations",  # aggregated list of all observe dinsertion events
        "ConstantChainInfoPerINN",  # lists all mutations observed in a single constant chain (each inn)
        "MutationsPerMotif",  # unique mutations events in each inn (EACH INN)
        "MutCountByRegion",  # simple statistic for mutations by gene/species
    ],

    supp_table_numbering=True,
    fail_on_missing_included_sheets=True
)
# # Only specific sheets
# export_constant_chain_analysis(result, "ConstantChainAnalysis.xlsx",
#     include_sheets=["MotifIndex", "CoMutationPairs", "UniqueSingleMutations"])

# # All except heavy tables
# export_constant_chain_analysis(result, "ConstantChainAnalysis.xlsx",
#     exclude_sheets=["FullSequenceList"])

# # Custom column serializers:
# # - str key → applied to that column in ALL sheets
# # - (sheet_name, col) tuple → only in that specific sheet
# export_constant_chain_analysis(result, "ConstantChainAnalysis.xlsx",
#     column_serializers={

#     }
# )

In [ ]:
import pickle
with open(results_path / 'results.pkl', 'wb') as o:
    pickle.dump(result, o)